In [1]:
import torch
import math
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torchvision import datasets, transforms
import torch.nn.functional as F
import time
import os
import torch.backends.cudnn as cudnn


os.environ["CUDA_VISIBLE_DEVICES"] = '0'                # GPU Number
start_time = time.time()
batch_size = 128
learning_rate = 0.001
root_dir = 'drive/app/cifar10/'
default_directory = 'drive/app/torch/save_models'

# Data Augmentation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),               # Random Position Crop
    transforms.RandomHorizontalFlip(),                  # right and left flip
    transforms.ToTensor(),                              # change [0,255] Int value to [0,1] Float value
    transforms.Normalize(mean=(0.4914, 0.4824, 0.4467), # RGB Normalize MEAN
                         std=(0.2471, 0.2436, 0.2616))  # RGB Normalize Standard Deviation
])

transform_test = transforms.Compose([
    transforms.ToTensor(),                              # change [0,255] Int value to [0,1] Float value
    transforms.Normalize(mean=(0.4914, 0.4824, 0.4467), # RGB Normalize MEAN
                         std=(0.2471, 0.2436, 0.2616))  # RGB Normalize Standard Deviation
])

# automatically download
train_dataset = datasets.CIFAR10(root=root_dir,
                                 train=True,
                                 transform=transform_train,
                                 download=True)

test_dataset = datasets.CIFAR10(root=root_dir,
                                train=False,
                                transform=transform_test)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True,            # at Training Procedure, Data Shuffle = True
                                           num_workers=4)           # CPU loader number

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False,            # at Test Procedure, Data Shuffle = False
                                          num_workers=4)            # CPU loader number


class AlexNet(nn.Module):

    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),   # Output Size : 64, 16, 16
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),                            # Output Size : 64, 8, 8
            nn.Conv2d(64, 192, kernel_size=3, padding=1),           # Output Size : 192, 8, 8
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),                            # Output Size : 192, 4, 4
            nn.Conv2d(192, 384, kernel_size=3, padding=1),          # Output Size : 384, 4, 4
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),          # Output Size : 256, 4, 4
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),          # Output Size : 256, 4, 4
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)                             # Output Size : 256, 2, 2
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256*2*2, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(1024, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = AlexNet()

optimizer = optim.SGD(model.parameters(), learning_rate,
                                momentum=0.9,
                                weight_decay=1e-4,
                                nesterov=True)
criterion = nn.CrossEntropyLoss()

if torch.cuda.device_count() > 0:
    print("USE", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model).cuda()
    cudnn.benchmark = True
else:
    print("USE ONLY CPU!")


def train(epoch):
    model.train()
    train_loss = 0
    total = 0
    correct = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        if torch.cuda.is_available():
            data, target = Variable(data.cuda()), Variable(target.cuda())
        else:
            data, target = Variable(data), Variable(target)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(output.data, 1)

        total += target.size(0)
        correct += predicted.eq(target.data).cpu().sum()
        if batch_idx % 10 == 0:
            print('Epoch: {} | Batch_idx: {} |  Loss: ({:.4f}) | Acc: ({:.2f}%) ({}/{})'
                  .format(epoch, batch_idx, train_loss / (batch_idx + 1), 100. * correct / total, correct, total))


def test():
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    for batch_idx, (data, target) in enumerate(test_loader):
        if torch.cuda.is_available():
            data, target = Variable(data.cuda()), Variable(target.cuda())
        else:
            data, target = Variable(data), Variable(target)

        outputs = model(data)
        loss = criterion(outputs, target)

        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += predicted.eq(target.data).cpu().sum()
    print('# TEST : Loss: ({:.4f}) | Acc: ({:.2f}%) ({}/{})'
          .format(test_loss / (batch_idx + 1), 100. * correct / total, correct, total))


def save_checkpoint(directory, state, filename='latest.tar.gz'):

    if not os.path.exists(directory):
        os.makedirs(directory)

    model_filename = os.path.join(directory, filename)
    torch.save(state, model_filename)
    print("=> saving checkpoint")

def load_checkpoint(directory, filename='latest.tar.gz'):

    model_filename = os.path.join(directory, filename)
    if os.path.exists(model_filename):
        print("=> loading checkpoint")
        state = torch.load(model_filename)
        return state
    else:
        return None

start_epoch = 0

checkpoint = load_checkpoint(default_directory)
if not checkpoint:
    pass
else:
    start_epoch = checkpoint['epoch'] + 1
    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])

for epoch in range(start_epoch, 165):

    if epoch < 80:
        lr = learning_rate
    elif epoch < 120:
        lr = learning_rate * 0.1
    else:
        lr = learning_rate * 0.01
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    train(epoch)
    save_checkpoint(default_directory, {
        'epoch': epoch,
        'model': model,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
    })
    test()

now = time.gmtime(time.time() - start_time)
print('{} hours {} mins {} secs for training'.format(now.tm_hour, now.tm_min, now.tm_sec))

100%|██████████| 170M/170M [00:13<00:00, 12.5MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


USE 1 GPUs!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch: 0 | Batch_idx: 0 |  Loss: (2.3041) | Acc: (8.59%) (11/128)
Epoch: 0 | Batch_idx: 10 |  Loss: (2.3024) | Acc: (9.73%) (137/1408)
Epoch: 0 | Batch_idx: 20 |  Loss: (2.3026) | Acc: (10.27%) (276/2688)
Epoch: 0 | Batch_idx: 30 |  Loss: (2.3027) | Acc: (9.68%) (384/3968)
Epoch: 0 | Batch_idx: 40 |  Loss: (2.3029) | Acc: (9.45%) (496/5248)
Epoch: 0 | Batch_idx: 50 |  Loss: (2.3029) | Acc: (9.18%) (599/6528)
Epoch: 0 | Batch_idx: 60 |  Loss: (2.3029) | Acc: (9.54%) (745/7808)
Epoch: 0 | Batch_idx: 70 |  Loss: (2.3029) | Acc: (9.73%) (884/9088)
Epoch: 0 | Batch_idx: 80 |  Loss: (2.3029) | Acc: (9.73%) (1009/10368)
Epoch: 0 | Batch_idx: 90 |  Loss: (2.3028) | Acc: (9.95%) (1159/11648)
Epoch: 0 | Batch_idx: 100 |  Loss: (2.3027) | Acc: (9.96%) (1288/12928)
Epoch: 0 | Batch_idx: 110 |  Loss: (2.3027) | Acc: (9.91%) (1408/14208)
Epoch: 0 | Batch_idx: 120 |  Loss: (2.3027) | Acc: (9.92%) (1537/15488)
Epoch: 0 | Batch_idx: 130 |  Loss: (2.3027) | Acc: (9.85%) (1652/16768)
Epoch: 0 | Batch_idx

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
# TEST : Loss: (0.9250) | Acc: (66.53%) (6653/10000)
Epoch: 46 | Batch_idx: 0 |  Loss: (0.8828) | Acc: (71.88%) (92/128)
Epoch: 46 | Batch_idx: 10 |  Loss: (0.9211) | Acc: (67.26%) (947/1408)
Epoch: 46 | Batch_idx: 20 |  Loss: (0.9305) | Acc: (66.41%) (1785/2688)
Epoch: 46 | Batch_idx: 30 |  Loss: (0.9404) | Acc: (65.55%) (2601/3968)
Epoch: 46 | Batch_idx: 40 |  Loss: (0.9483) | Acc: (65.38%) (3431/5248)
Epoch: 46 | Batch_idx: 50 |  Loss: (0.9520) | Acc: (65.35%) (4266/6528)
Epoch: 46 | Batch_idx: 60 |  Loss: (0.9565) | Acc: (65.14%) (5086/7808)
Epoch: 46 | Batch_idx: 70 |  Loss: (0.9556) | Acc: (65.20%) (5925/9088)
Epoch: 46 | Batch_idx: 80 |  Loss: (0.9538) | Acc: (65.24%) (6764/10368)
Epoch: 46 | Batch_idx: 90 |  Loss: (0.9540) | Acc: (65.16%) (7590/11648)
Epoch: 46 | Batch_idx: 100 |  Loss: (0.9548) | Acc: (65.18%) (8426/12928)
Epoch: 46 | Batch_idx: 110 |  Loss: (0.9457) | Acc: (65.57%) (9316/14208)
Epoch: 46 | Batch_idx: 120 |  Loss: (0.9466) |